# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
# [replaced classroom cell] The Colab/zip loader and `!pip install` are not needed here:
# the client pack lives in source_systems/ and dependencies come from requirements.txt.
import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "source_systems").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
BASE = ROOT / "source_systems"                    # read-only client systems
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "Challenges" / "output"             # scratch outputs of the challenge notebooks
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)
print("BASE:", BASE, "| database exists:", (BASE / "database" / "flasheats.db").exists())

BASE: B:\Trimester-9\FDE\assignment-2\source_systems | database exists: True


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)

# TODO: build_order_timeline(order_id)

late O00001 on-time O00003 intervention O00781


In [4]:
# Build the timeline from EVERY source (not only the orders table) and sort by time
pages = sorted(max((ROOT / "data" / "raw").glob("run_*/api"), key=lambda p: p.stat().st_mtime).glob("dispatch_page_*.json"))
dispatch = pd.DataFrame([r for p in pages for r in json.loads(p.read_text())["payload"]["data"]])     # retrieved + preserved by the pipeline
with open(BASE / "data" / "driver_events.json") as f:
    dev = pd.DataFrame([{"driver_id": d["driver_id"], **e} for d in json.load(f) for e in d["events"]])
rstat = pd.read_csv(BASE / "data" / "restaurant_status.csv")
T = lambda s: pd.to_datetime(s, format="mixed", errors="coerce")

def build_order_timeline(order_id):
    ev = []
    o = base_orders[base_orders.order_id == order_id].iloc[0]
    for col, et in [("created_at", "ORDER_CREATED"), ("pickup_at", "PICKED_UP"), ("actual_delivery_at", "DELIVERED")]:
        ev.append((o[col], et, "customer" if et == "ORDER_CREATED" else "driver", "orders DB"))
    d = dispatch[dispatch.order_id == order_id]
    for _, r in d.iterrows():
        ev += [(r.assigned_at, "DRIVER_ASSIGNED " + str(r.original_driver_id), "dispatch", "Dispatch API"),
               (r.estimated_pickup_at, "PICKUP_ESTIMATED", "dispatch", "Dispatch API"), (r.reassigned_at, "DRIVER_REASSIGNED " + str(r.driver_id), "dispatch", "Dispatch API")]
    ev += [(r.timestamp, "DRIVER_APP_" + r.type.upper(), "driver " + r.driver_id, "driver app") for r in dev[dev.order_id == order_id].itertuples()]
    ev += [(r.last_updated_at, "RESTAURANT_" + r.status.strip().upper(), "restaurant", "restaurant feed") for r in rstat[rstat.order_id == order_id].itertuples()]
    ev += [(r.action_at, "APP_" + r.action_type, "customer", "customer app") for r in customer_actions[customer_actions.order_id == order_id].itertuples()]
    ev += [(r.created_at, "SUPPORT_TICKET " + r.category.strip(), "customer", "support desk") for r in tickets[tickets.order_id == order_id].itertuples()]
    ev += [(r.intervention_at, "INTERVENTION_" + r.intervention_type, r.initiated_by, "interventions log") for r in interventions[interventions.order_id == order_id].itertuples()]
    t = pd.DataFrame(ev, columns=["event_time", "event_type", "actor", "source_system"])
    t["event_time"] = T(t.event_time)
    return t.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)

promised = base_orders.set_index("order_id").promised_eta
for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("WITH INTERVENTION", intervention_order)]:
    print(f"=== {label}: {oid} | promised {promised[oid]}"); display(build_order_timeline(oid))

=== ON TIME: O00003 | promised 2026-08-01T20:17:52.941967

,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer,orders DB
1,2026-08-01 19:35:00.002443,DRIVER_ASSIGNED D039,dispatch,Dispatch API
2,2026-08-01 19:35:00.002443,DRIVER_APP_ASSIGNED,driver D039,driver app
3,2026-08-01 19:44:37.017130,DRIVER_APP_GPS_PING,driver D039,driver app
4,2026-08-01 19:48:00.000000,RESTAURANT_PREPARING,restaurant,restaurant feed
5,2026-08-01 19:53:00.000000,PICKUP_ESTIMATED,dispatch,Dispatch API
6,2026-08-01 19:54:14.031817,DRIVER_APP_GPS_PING,driver D039,driver app
7,2026-08-01 19:57:50.820366,PICKED_UP,driver,orders DB
8,2026-08-01 19:57:50.820366,DRIVER_APP_PICKED_UP,driver D039,driver app
9,2026-08-01 20:03:51.046504,DRIVER_APP_GPS_PING,driver D039,driver app


=== LATE: O00001 | promised 2026-08-13T14:11:36


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer,orders DB
1,2026-08-13 13:01:09.659644,DRIVER_ASSIGNED D103,dispatch,Dispatch API
2,2026-08-13 13:01:09.659644,DRIVER_APP_ASSIGNED,driver D103,driver app
3,2026-08-13 13:08:00.000000,INTERVENTION_DRIVER_REASSIGNMENT,dispatch,interventions log
4,2026-08-13 13:14:00.000000,PICKUP_ESTIMATED,dispatch,Dispatch API
5,2026-08-13 13:14:42.222353,DRIVER_APP_GPS_PING,driver D103,driver app
6,2026-08-13 13:21:00.000000,APP_ETA_VIEWED,customer,customer app
7,2026-08-13 13:28:14.785062,DRIVER_APP_GPS_PING,driver D103,driver app
8,2026-08-13 13:35:49.825561,PICKED_UP,driver,orders DB
9,2026-08-13 13:35:49.825561,DRIVER_APP_PICKED_UP,driver D103,driver app


=== WITH INTERVENTION: O00781 | promised 2026-08-22T13:48:36


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer,orders DB
1,2026-08-22 12:40:42.931820,DRIVER_ASSIGNED D103,dispatch,Dispatch API
2,2026-08-22 12:40:42.931820,DRIVER_APP_ASSIGNED,driver D103,driver app
3,2026-08-22 12:51:00.000000,PICKUP_ESTIMATED,dispatch,Dispatch API
4,2026-08-22 12:54:00.000000,APP_ETA_VIEWED,customer,customer app
5,2026-08-22 12:57:00.000000,INTERVENTION_PRIORITY_DISPATCH,operations,interventions log
6,2026-08-22 13:04:00.000000,RESTAURANT_READY,restaurant,restaurant feed
7,2026-08-22 13:07:42.051142,DRIVER_APP_GPS_PING,driver D103,driver app
8,2026-08-22 13:11:17.309762,PICKED_UP,driver,orders DB
9,2026-08-22 13:11:17.309762,DRIVER_APP_PICKED_UP,driver D103,driver app


> ✅ **Answer**

- **Where the time goes:** the gap between `DRIVER_ASSIGNED` and `PICKED_UP` is where the time is lost, and nothing records what happens inside it. There is no arrival event, so the time spent waiting at the restaurant and the ride there form one block.
- **Late order O00001:** the customer opened support at 14:20, before the 14:11 promise was broken, and the reassignment intervention came during the pre-pickup gap.
- **Two systems disagree on who delivered:** after a reassignment, the orders DB and Dispatch name different drivers.
- **Pipeline cross-check:** the pipeline builds the same lifecycle for all 1600 orders (`data/processed/fact_event.csv`, 21 518 events from 7 systems).

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [5]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# TODO: document grain + relationships

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


> ✅ **Answer**

| Table (pipeline) | Grain: one row per… | Primary key | Foreign keys | Supports |
|---|---|---|---|---|
| `dim_customer` | customer | `customer_id` | — | customer → orders |
| `fact_order` | order | `order_id` | customer, restaurant, driver (**final, from Dispatch**) | order → outcome, stage durations, dq flags, KPI population |
| `fact_event` | lifecycle event | (order, time, type, source) | `order_id` | order → events and states |
| `fact_interaction` | app action or ticket | `interaction_id` | `order_id`, `customer_id` | order → customer and support interactions |
| `fact_intervention` | intervention | `intervention_id` | `order_id` | order → interventions |
| `order_journey` | order | `order_id` | as `fact_order` | interaction → intervention → outcome in one row |

**Why not mirror every source table?**
- The sources disagree on grain (1603 vs 1600 orders), on spelling, and even on the driver.
- The model is organised around the workflow the KPI lives in. One-to-many tables are aggregated *before* joining, so the order count cannot inflate.
- Every business question becomes one filter on `order_journey`.

Diagrams: `docs/data-model.md`.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [6]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# TODO: add flags, aggregate interventions, and join to outcomes/orders

In [7]:
flags = customer_actions.groupby("order_id").agg(support_opened=("action_type", lambda s: (s == "SUPPORT_OPENED").any()),
                                                  cancel_attempted=("action_type", lambda s: (s == "CANCEL_ATTEMPTED").any()),
                                                  eta_views=("action_type", lambda s: int((s == "ETA_VIEWED").sum())))
tk = tickets.dropna(subset=["order_id"]).drop_duplicates().groupby("order_id").size().rename("tickets")
iv = interventions.groupby("order_id").agg(intervention_count=("intervention_id", "count"), intervention_types=("intervention_type", lambda s: ";".join(sorted(set(s)))))
journey = (base_orders[["order_id", "customer_id", "final_status"]]
           .merge(flags, on="order_id", how="left").merge(tk, on="order_id", how="left").merge(iv, on="order_id", how="left")
           .merge(outcomes[["order_id", "late_flag", "delay_min"]], on="order_id", how="left"))
assert len(journey) == base_orders.order_id.nunique(), "aggregate BEFORE joining - the order count must not change"
journey = journey.fillna({"support_opened": False, "cancel_attempted": False, "eta_views": 0, "tickets": 0, "intervention_count": 0, "intervention_types": ""})
journey["support_interaction"] = journey.support_opened.astype(bool) | (journey.tickets > 0)
display(journey.head())
late = journey[journey.late_flag == 1]
print("1. late orders with a support interaction:", int(late.support_interaction.sum()), "of", len(late))
print("2. orders that received an intervention:", int((journey.intervention_count > 0).sum()))
print("3. most common intervention:", interventions.intervention_type.value_counts().head(1).to_dict())
frus = journey[journey.support_interaction & (journey.intervention_count == 0)]
print("4. frustrated journeys with no intervention:", len(frus), "| of which late:", int((frus.late_flag == 1).sum()))
display(frus.sort_values("delay_min", ascending=False).head(5)[["order_id", "support_opened", "tickets", "delay_min"]])
oj = pd.read_csv(ROOT / "data" / "processed" / "order_journey.csv")
print("reconciliation with the pipeline's order_journey: rows", len(oj), "vs", len(journey),
      "| orders with intervention", int((oj.intervention_count > 0).sum()), "vs", int((journey.intervention_count > 0).sum()))

,order_id,customer_id,final_status,support_opened,cancel_attempted,eta_views,tickets,intervention_count,intervention_types,late_flag,delay_min,support_interaction
0,O00001,C0168,delivered,True,False,1.0,0.0,1.0,DRIVER_REASSIGNMENT,1.0,10.82,True
1,O00002,C0043,delivered,False,False,0.0,0.0,0.0,,1.0,3.97,False
2,O00003,C0855,delivered,False,False,0.0,0.0,0.0,,0.0,-4.41,False
3,O00004,C0229,cancelled,False,False,0.0,0.0,0.0,,NaN,NaN,False
4,O00005,C0040,delivered,False,False,0.0,0.0,0.0,,1.0,21.02,False


1. late orders with a support interaction: 258 of 843
2. orders that received an intervention: 430
3. most common intervention: {'DRIVER_REASSIGNMENT': 155}
4. frustrated journeys with no intervention: 218 | of which late: 191


,order_id,support_opened,tickets,delay_min
193,O00194,False,1.0,38.34
736,O00737,True,0.0,36.28
302,O00303,False,1.0,33.16
210,O00211,True,1.0,32.38
1234,O01235,False,1.0,32.00


reconciliation with the pipeline's order_journey: rows 1600 vs 1600 | orders with intervention 430 vs 430


> ✅ **Answer**

- **Late orders with a support interaction: 258 of 843.** This uses the client's `order_outcomes` file. The pipeline's validated population gives 257 of 837, and the small difference is the 9 chronology-rule orders.
- **Orders that received an intervention: 430.**
- **Most common intervention:** `DRIVER_REASSIGNMENT`, with 155.
- **Frustrated journeys with no intervention: 218**, most of them late. These are the ops team's clearest missed opportunities.
- **Reconciliation:** the notebook table and the pipeline's `order_journey` agree on both row count and intervention count.

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [8]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

In [9]:
v = journey[journey.late_flag.notna()]
m = pd.DataFrame([
    ("M1 late delivery rate", "outcome", round(100 * v.late_flag.mean(), 2), "late / delivered-with-time"),
    ("M2 median lateness (late orders)", "outcome", round(v[v.late_flag == 1].delay_min.median(), 1), "min"),
    ("M4 support contact rate on late orders", "interaction", round(100 * v[v.late_flag == 1].support_interaction.mean(), 1), "% of late orders"),
    ("M4a support contact rate on on-time orders", "interaction", round(100 * v[v.late_flag == 0].support_interaction.mean(), 1), "baseline"),
    ("M5 intervention coverage", "intervention", round(100 * (v.intervention_count > 0).mean(), 1), "% of delivered orders"),
], columns=["metric", "type", "value", "unit"])
display(m)
display(pd.read_csv(ROOT / "output" / "metrics.csv")[["metric_id", "metric", "category", "value", "unit"]].head(18))

,metric,type,value,unit
0,M1 late delivery rate,outcome,56.39,late / delivered-with-time
1,M2 median lateness (late orders),outcome,8.00,min
2,M4 support contact rate on late orders,interaction,30.60,% of late orders
3,M4a support contact rate on on-time orders,interaction,5.20,baseline
4,M5 intervention coverage,intervention,27.00,% of delivered orders


,metric_id,metric,category,value,unit
0,M1,Late delivery rate (validated population),outcome,56.33,%
1,M1a,Late delivery rate (historical dashboard definition),outcome,56.39,%
2,M1b,Meaningfully late rate (> 10 min),outcome,23.15,%
3,M1c,Late delivery rate (sensitivity: back-fill missing delivery time from driver telemetry),outcome,56.53,%
4,M2,Median lateness among late orders,outcome,8.0,min
5,M2a,P90 delay across all validated deliveries,outcome,17.9,min
6,M2b,Worst single delay,outcome,50.3,min
7,M3,Share of lateness accumulated before pickup (vs in transit),workflow,98.91,%
8,M3a,Median pre-pickup overrun on late orders,workflow,13.9,min
9,M3b,Median transit overrun on late orders,workflow,-5.9,min


> ✅ **Answer**

| # | Metric | Formula | Grain | Why it matters | Link to the KPI |
|---|---|---|---|---|---|
| M1 | Late delivery rate (**outcome**) | late / validated deliveries | order | the KPI itself | direct |
| M2 | Lateness severity (**outcome**) | median delay of late orders; P90 | order | many small misses or a few disasters? | severity |
| M3 | Share of lateness before pickup (**workflow**) | pre-pickup overrun / all stage overrun | order → stage | tells ops *where* to act | the driver of the KPI |
| M4 | Support contact rate on late orders (**interaction**) | late with ticket or SUPPORT_OPENED / late | order | the customer impact | consequence of the KPI |
| M5 | Intervention coverage and late rate with / without (**intervention**) | intervened / orders | order | does ops reach the right orders? | the lever |

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [10]:
# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation

In [11]:
def rate(d): return round(100 * (d.late_flag == 1).mean(), 1) if len(d) else None
print("A. median delay with / without a support interaction:", v[v.support_interaction].delay_min.median(), "/", v[~v.support_interaction].delay_min.median(),
      "| late rate:", rate(v[v.support_interaction]), "/", rate(v[~v.support_interaction]))
print("B. late rate with / without an intervention:", rate(v[v.intervention_count > 0]), "/", rate(v[v.intervention_count == 0]))
c = interventions.merge(v[["order_id", "late_flag"]], on="order_id").groupby("intervention_type").late_flag.agg(orders="size", late_rate=lambda s: round(100 * s.mean(), 1))
print("C. late rate by intervention type:"); display(c.sort_values("late_rate"))
fr = pd.read_csv(ROOT / "output" / "insights" / "fair_ranking_restaurants.csv")
print("D. restaurants by late orders, with 95% Wilson intervals (beyond the class):")
display(fr.sort_values("late_orders", ascending=False).head(8)[["restaurant_id", "orders", "late_orders", "late_rate_pct", "ci_low_pct", "ci_high_pct", "verdict"]])
e = v[v.support_interaction & (v.intervention_count > 0) & (v.late_flag == 1)]
print("E. support + intervention + still late:", len(e)); display(e.sort_values("delay_min", ascending=False).head(5)[["order_id", "intervention_types", "delay_min"]])

A. median delay with / without a support interaction: 12.285 / -0.25 | late rate: 88.4 / 48.6
B. late rate with / without an intervention: 56.4 / 56.4
C. late rate by intervention type:


,orders,late_rate
intervention_type,,
PRIORITY_DISPATCH,86,51.2
RESTAURANT_CONTACT,108,52.8
DRIVER_REASSIGNMENT,147,57.8
CUSTOMER_CREDIT,63,66.7


D. restaurants by late orders, with 95% Wilson intervals (beyond the class):


,restaurant_id,orders,late_orders,late_rate_pct,ci_low_pct,ci_high_pct,verdict
8,R050,35,22,62.9,46.3,76.8,not distinguishable from fleet
11,R023,32,20,62.5,45.3,77.1,not distinguishable from fleet
2,R051,29,20,69.0,50.8,82.7,not distinguishable from fleet
60,R024,25,20,80.0,60.9,91.1,worse than fleet (significant)
5,R004,29,19,65.5,47.3,80.1,not distinguishable from fleet
15,R011,31,19,61.3,43.8,76.3,not distinguishable from fleet
7,R045,29,19,65.5,47.3,80.1,not distinguishable from fleet
30,R030,37,19,51.4,35.9,66.6,not distinguishable from fleet


E. support + intervention + still late: 67


,order_id,intervention_types,delay_min
100,O00101,CUSTOMER_CREDIT,85.51
197,O00198,RESTAURANT_CONTACT,35.23
496,O00497,PRIORITY_DISPATCH,31.78
612,O00613,PRIORITY_DISPATCH,27.18
7,O00008,CUSTOMER_CREDIT,26.86


> ✅ **Answer**

- **A.** Orders with a support interaction are far more often late: 88 % against 49 %. *Tells:* complaints are a reliable lagging signal. *Does not prove:* that support contact causes delay; the delay causes the contact.
- **B.** The late rate is about 56 % with or without an intervention. *Tells:* interventions are not associated with a lower late rate. *Does not prove:* that they are useless. They target at-risk orders, a selection effect, and the interventions log does not reconcile with Dispatch.
- **C.** `PRIORITY_DISPATCH` has the lowest late rate, about 51 %, and `CUSTOMER_CREDIT` the highest, about 67 %. *Tells:* credits are handed out after a failure. *Does not prove:* which action works, because each type has a different trigger.
- **D.** The naive "most late orders" list names ten restaurants, but only **R024** is statistically worse than the fleet. *Tells:* lateness is systemic. *Does not prove:* kitchen fault, because preparation time is not observed.
- **E.** 67 journeys had a support interaction and an intervention and were still late (66 on the pipeline's validated population). *Tells:* the recovery loop acts too late or too weakly. *Does not prove:* that earlier action would have saved them. The pipeline's early-warning backtest tests exactly that timing question.

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

> ✅ **Answer**

```
PROJECT KPI        reduce late delivery rate (M1)
  ↑ OUTCOMES       M1 late rate · M2 severity · M4 customer contact
  ↑ WORKFLOW       M3 pre-pickup overrun share · dispatch wait · assign→pickup · transit · Dispatch pickup-estimate bias
  ↑ INTERVENTIONS  M5 coverage and type · NEW: pickup-overrun trigger (alert at estimated pickup + k min)
  ↑ SOURCES/EVENTS orders DB · Dispatch API · driver app · restaurant feed · customer app · support desk · interventions log · Open-Meteo (external check)
```
1. **Controllable by operations:** dispatch wait, intervention coverage and timing (M5), which orders receive an intervention, and the pickup estimate behind the promise.
2. **Outcomes:** M1, M2 and M4.
3. **The missing event that limits the model most:** *driver arrived at restaurant*. It cannot be inferred from GPS either (see the Class 5 Starter).
4. **Instrument next:** an arrival tap, a mandatory delivery timestamp, reassignment written back to the orders table, a cancellation timestamp, and the provenance of `weather_bucket`.

**Beyond the class, what ops can do on Monday:** the pipeline backtested a transparent trigger, *"not picked up 10 minutes after Dispatch's own estimate"*. It was chosen on 1–21 August and proven on 22–28 August, where it reached **96 % precision and 72 % recall, about 45 minutes before the promise breaks**. Today, 446 of the late orders it would have caught received no intervention at all. See `output/decision_memo.md`.

**Modelling limitation:** the pre-pickup stage is a black box until the arrival event exists.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.